# QSage Tutorial

This tutorial demonstrates how to use **Quantum Sage (QSage)**, a meta-learning tool that predicts which quantum or classical ML model will perform best on your dataset.

## What is QSage?

QSage is a **surrogate model** trained on extensive benchmarking data that:

- **Predicts model performance** without running expensive experiments
- **Recommends best models** based on dataset characteristics
- **Saves computational resources** by avoiding trial-and-error
- **Supports both quantum and classical** ML algorithms

## 1. Setup and Imports

In [ ]:
import pandas as pd
import os
import pickle

# Import QSage
from apps.sage.sage import QuantumSage

print("✓ Imports successful")

## 2. Load Benchmark Data

QSage requires benchmark data containing:
- Dataset complexity features (# Features, # Samples, Intrinsic_Dimension, etc.)
- Performance metrics (accuracy, f1_score, auc)
- Metadata (Dataset, embeddings, model, etc.)

**Download the data**: [https://ibm.biz/QSageModel](https://ibm.biz/QSageModel)

In [ ]:
# Set paths
dir_results = 'results'  # Directory containing downloaded files
file_input = os.path.join(dir_results, 'Compiled_QMLBench_results.csv')

# Load benchmark results
results_df = pd.read_csv(file_input)

print(f"Loaded {len(results_df)} benchmark results")
print(f"Datasets: {results_df['Dataset'].nunique()}")
print(f"Models: {results_df['model'].nunique()}")
print(f"\nFirst few rows:")
results_df.head()

## 3. Initialize QSage

**Important**: The current QSage API uses `data_input` parameter (not `data`, `features`, `metrics`, `sage_type`).

In [ ]:
# Select a held-out dataset for testing
held_out_dataset = 'iris_binary_class_dataset'

# Split data into training and held-out
train_df = results_df[~results_df['Dataset'].str.contains(held_out_dataset)]
held_out_df = results_df[results_df['Dataset'].str.contains(held_out_dataset)]

print(f"Training data: {len(train_df)} results")
print(f"Held-out data: {len(held_out_df)} results")

# Initialize QSage with correct API
sage = QuantumSage(data_input=train_df)

print(f"\n✓ QSage initialized")
print(f"Available models: {sage._available_models}")
print(f"Available metrics: {sage._available_metrics}")

## 4. Train QSage Sub-Sages

Train surrogate models for each ML model and metric combination.

In [ ]:
# Train sub-sages with Random Forest
print("Training QSage sub-sages...")
sage.train_sub_sages(
    test_size=0.2,
    sage_type='random_forest',  # or 'mlp'
    n_iter=50,  # Number of hyperparameter search iterations
    cv=5  # Cross-validation folds
)
print("✓ Training complete!")

## 5. Visualize Training Results

In [ ]:
# Plot training results
sage.plot_results(figsize=(8, 5))

## 6. Make Predictions

Use trained QSage to predict model performance on new data.

In [ ]:
# Prepare held-out dataset features
held_out_features = held_out_df[sage._columns_data_features].drop_duplicates()

print("Held-out dataset characteristics:")
print(held_out_features)

# Make predictions for accuracy metric
predictions = sage.predict(held_out_features, metric='accuracy')

print(f"\n✓ Generated predictions for {len(predictions)} models")
print("\nTop 10 Predicted Models:")
print(predictions.head(10))

## 7. Compare Predictions vs Actual Results

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Get actual results for comparison
actual_results = held_out_df.groupby('model')['accuracy'].mean().reset_index()
actual_results.columns = ['model', 'actual_accuracy']

# Merge with predictions
comparison = predictions.merge(actual_results, on='model')

# Plot
plt.figure(figsize=(10, 6))
plt.scatter(comparison['accuracy'], comparison['actual_accuracy'], alpha=0.6, s=100)
plt.plot([0, 1], [0, 1], 'r--', label='Perfect prediction')
plt.xlabel('Predicted Accuracy')
plt.ylabel('Actual Accuracy')
plt.title('QSage Prediction Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Calculate error
mae = abs(comparison['accuracy'] - comparison['actual_accuracy']).mean()
print(f"\nMean Absolute Error: {mae:.4f}")

## 8. Save Trained QSage Model

In [ ]:
# Save trained model
file_sage = os.path.join(dir_results, 'my_qsage_model.pkl')
with open(file_sage, 'wb') as f:
    pickle.dump(sage, f)
print(f"✓ Model saved to {file_sage}")

## 9. Load Pre-trained Model (Optional)

In [ ]:
# Load pre-trained QSage model
with open(file_sage, 'rb') as f:
    loaded_sage = pickle.load(f)

print("✓ Pre-trained QSage model loaded")
print(f"Available models: {loaded_sage._available_models}")
print(f"Available metrics: {loaded_sage._available_metrics}")

## Summary

In this tutorial, you learned how to:

1. ✅ Load benchmark data with model performance results
2. ✅ Initialize QSage with the correct API (`data_input` parameter)
3. ✅ Train QSage sub-sages using `train_sub_sages()` method
4. ✅ Make predictions for new datasets using `predict()` method
5. ✅ Visualize and compare predictions vs actual results
6. ✅ Save and load trained QSage models

## Key API Points

- **Initialization**: `QuantumSage(data_input=df)` - only takes `data_input` parameter
- **Training**: `sage.train_sub_sages(sage_type='random_forest')` - specify model type here
- **Prediction**: `sage.predict(features, metric='accuracy')` - predict for specific metric
- **Visualization**: `sage.plot_results()` - plot training performance

## Next Steps

- Use QSage on your own benchmark data
- Combine with QProfiler for comprehensive model evaluation
- Experiment with different sage types (Random Forest vs MLP)
- Try different metrics (accuracy, f1_score, auc)

## See Also

- [QSage Documentation](../../apps/sage.rst) - Full API reference
- [QProfiler Tutorial](../QProfiler/example_qprofiler.ipynb) - Benchmark models
- [Data Generation Tutorial](../Artificial_data_generation/example_data_generation.ipynb) - Create test datasets